# Sacred -> Techno v2 backend (Colab Pro)

LoRA 파인튜닝 모델 + 보컬 chop + 드럼 합성 + 사이드체인 믹스 방식의 v2 파이프라인을 실행합니다.
프론트엔드 UI는 그대로이고, `/convert` 엔드포인트 계약도 동일합니다.

## 사전 설정 (한 번만)

1. https://dashboard.ngrok.com 에 가입하고 Auth Token을 확인합니다.
2. https://dashboard.ngrok.com/domains 에서 고정 도메인(static domain)을 발급받습니다.
3. Colab 왼쪽 사이드바 열쇠 아이콘(Secrets)에 다음을 등록하고 'Notebook access'를 켜주세요.
   - `NGROK_AUTH_TOKEN`: ngrok 토큰
   - `NGROK_DOMAIN`: 고정 도메인 (`https://` 제외, 예: `your-name.ngrok-free.app`)
4. Vercel `NEXT_PUBLIC_ML_BACKEND_URL`을 `https://<고정 도메인>`으로 설정하고 재배포합니다 (한 번만).

**LoRA adapter**는 서버 기동 시 자동으로 Google Drive에서 다운로드됩니다. 별도 설정 불필요.

## 발표 직전에 할 일

1. 런타임 -> 런타임 유형 변경 -> GPU (L4 권장)
2. 아래 셀을 순서대로 실행
3. 마지막 셀에 'Backend URL'이 출력되고 고정 도메인과 일치하면 완료
4. 노트북을 끄지 말고 발표가 끝날 때까지 켜둔 상태로 유지

In [ ]:
import os

if not os.path.exists("/content/MIR"):
    !git clone https://github.com/rocgwang/MIR.git /content/MIR
else:
    !cd /content/MIR && git pull

%cd /content/MIR/backend_v2

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyngrok

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
NGROK_DOMAIN = userdata.get("NGROK_DOMAIN")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
import subprocess
import time
import urllib.request

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/MIR/backend_v2",
)

print("서버와 모델 로딩을 기다리는 중입니다.")
print("처음 실행할 때는 MusicGen 모델 + LoRA adapter 다운로드 때문에 몇 분 걸릴 수 있습니다.")
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/openapi.json", timeout=2)
        break
    except Exception:
        time.sleep(5)
print("서버 준비 완료")

In [ ]:
tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN)
print("Backend URL:", tunnel.public_url)
print("이 URL이 Vercel의 NEXT_PUBLIC_ML_BACKEND_URL 값과 같은지 확인하세요.")

## 발표 종료 후

발표가 끝나면 '런타임 > 런타임 연결 해제 및 삭제'로 종료해서 Colab Pro 사용량을 절약하세요.